<a href="https://colab.research.google.com/github/SrustiUB/Data-Science/blob/main/Copulas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance scipy pandas numpy matplotlib seaborn statsmodels

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew, kurtosis, jarque_bera

In [ ]:
nifty = yf.download("^NSEI",
                    start="2018-01-01",
                    end="2025-01-01")

reliance = yf.download("RELIANCE.NS",
                       start="2018-01-01",
                       end="2025-01-01")

/tmp/ipykernel_545/737645467.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nifty = yf.download("^NSEI",
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_545/737645467.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  reliance = yf.download("RELIANCE.NS",
[*********************100%***********************]  1 of 1 completed


In [ ]:
print(nifty.columns)

MultiIndex([( 'Close', '^NSEI'),
            (  'High', '^NSEI'),
            (   'Low', '^NSEI'),
            (  'Open', '^NSEI'),
            ('Volume', '^NSEI')],
           names=['Price', 'Ticker'])


In [ ]:
prices = pd.concat([
    nifty[("Close", "^NSEI")].rename("NIFTY"),
    reliance[("Close", "RELIANCE.NS")].rename("RELIANCE")
], axis=1)

prices = prices.dropna()

prices.head()

,NIFTY,RELIANCE
Date,,
2018-01-02,10442.200195,400.630890
2018-01-03,10443.200195,402.235809
2018-01-04,10504.799805,404.654114
2018-01-05,10558.849609,405.951202
2018-01-08,10623.599609,408.281616


In [ ]:
returns = np.log(prices / prices.shift(1))
returns = returns.dropna()

returns.head()

,NIFTY,RELIANCE
Date,,
2018-01-03,0.000096,0.003998
2018-01-04,0.005881,0.005994
2018-01-05,0.005132,0.003200
2018-01-08,0.006114,0.005724
2018-01-09,0.001261,0.013266


In [ ]:
returns.describe()

,NIFTY,RELIANCE
count,1722.000000,1722.000000
mean,0.000475,0.000640
std,0.011223,0.018207
min,-0.139038,-0.141032
25%,-0.004530,-0.008854
50%,0.000892,0.000715
75%,0.006494,0.009718
max,0.084003,0.137307


In [ ]:
from scipy.stats import skew, kurtosis

print("Skewness")
print(returns.apply(skew))

print("\nKurtosis")
print(returns.apply(kurtosis))

Skewness
NIFTY      -1.535975
RELIANCE    0.041820
dtype: float64

Kurtosis
NIFTY       21.929586
RELIANCE     9.079361
dtype: float64


In [ ]:
print(returns.corr(method="pearson"))
print(returns.corr(method="spearman"))
print(returns.corr(method="kendall"))

             NIFTY  RELIANCE
NIFTY     1.000000  0.686294
RELIANCE  0.686294  1.000000
             NIFTY  RELIANCE
NIFTY     1.000000  0.640055
RELIANCE  0.640055  1.000000
             NIFTY  RELIANCE
NIFTY     1.000000  0.466595
RELIANCE  0.466595  1.000000


In [ ]:
from scipy.stats import rankdata

u = rankdata(returns["NIFTY"]) / (len(returns) + 1)
v = rankdata(returns["RELIANCE"]) / (len(returns) + 1)

In [ ]:
from scipy.stats import kendalltau

tau, p = kendalltau(
    returns["NIFTY"],
    returns["RELIANCE"]
)

print("Kendall Tau =", tau)
print("p-value =", p)

Kendall Tau = 0.46659459123851643
p-value = 4.3920353302652e-185


In [ ]:
import numpy as np

rho = np.sin(np.pi * tau / 2)

print("Gaussian Copula rho =", rho)

Gaussian Copula rho = 0.6690464663214865


In [ ]:
from scipy.stats import norm

z1 = norm.ppf(u)
z2 = norm.ppf(v)

In [ ]:
corr = np.corrcoef(z1, z2)

print(corr)

[[1.         0.65963225]
 [0.65963225 1.        ]]


In [ ]:
from scipy.stats import multivariate_normal

rho = corr[0,1]

Sigma = np.array([[1, rho],
                  [rho, 1]])

loglik = np.sum(
    multivariate_normal(mean=[0,0], cov=Sigma).logpdf(np.column_stack((z1,z2)))
    - norm.logpdf(z1)
    - norm.logpdf(z2)
)

print("Log-Likelihood =", loglik)

Log-Likelihood = 491.7451634179971


In [ ]:
k = 1
n = len(z1)

AIC = -2*loglik + 2*k
BIC = -2*loglik + np.log(n)*k

print("AIC =", AIC)
print("BIC =", BIC)

AIC = -981.4903268359942
BIC = -976.0390851510066


In [ ]:
results = []

results.append({
    "Copula":"Gaussian",
    "LogLik":loglik,
    "AIC":AIC,
    "BIC":BIC
})

results

[{'Copula': 'Gaussian',
  'LogLik': np.float64(491.7451634179971),
  'AIC': np.float64(-981.4903268359942),
  'BIC': np.float64(-976.0390851510066)}]